In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from constants import UCI_CLIENT_SITES_TO_VALIDATE

In [ ]:
def mae(y_true: np.array, y_hat: np.array) -> float:
    return np.mean(np.abs(y_true - y_hat))


def rmsse(y_true: np.array, y_hat: np.array, season: int = 1) -> float:
    """Root Mean Square Scaled Error"""
    mse = np.mean((y_true - y_hat) ** 2)
    naive_errors =  np.mean((y_true[season:] - y_true[:-season]) ** 2)
    return np.sqrt(mse / naive_errors)

In [ ]:
period = 24
n_timestamps = 100
x = np.arange(n_timestamps)
y = np.sin(2 * np.pi * x / period) + np.random.normal(loc=0, scale=0.2, size=(n_timestamps,))

plt.plot(x, y)
plt.plot(x + period, y)

In [ ]:
RESULTS_DIR = Path("../../results/uci")
MODELS = ["naive", "ets", "tcn"]
SEASONAL_PERIOD = 96

n_folds = 10
model_errors = {}
for model in MODELS:
    site_errors = {}
    for site in UCI_CLIENT_SITES_TO_VALIDATE:
        model_site_dir = RESULTS_DIR / model / site
        
        fold_errors = []
        for fold in range(n_folds):
            forecasts_file = model_site_dir / f"forecasts_{site}_fold_{fold}.pq"
            forecasts_df = pl.read_parquet(forecasts_file)

            y_true = forecasts_df["demand"].to_numpy()
            y_hat = forecasts_df["forecast"].to_numpy()
            error = mae(y_true, y_hat)
            fold_errors.append(error)

        site_errors[site] = fold_errors
    model_errors[model] = site_errors


In [ ]:
fig, axes = plt.subplots(len(UCI_CLIENT_SITES_TO_VALIDATE), 1, figsize=(10, 2.5 * len(UCI_CLIENT_SITES_TO_VALIDATE)))

colors = ["tab:blue", "tab:orange", "tab:green"]
for m, model in enumerate(MODELS):
    for i, site in enumerate(UCI_CLIENT_SITES_TO_VALIDATE):
        errors = model_errors[model][site]
        axes[i].plot(np.arange(n_folds), errors, label=model, color=colors[m], lw=0.75, ls="--", marker="o")
        axes[i].axhline(np.mean(errors), color=colors[m], ls="--", lw=0.75)

        axes[i].legend()
        axes[i].set(title=site, ylabel="MAE", xlabel="Fold" if i == len(UCI_CLIENT_SITES_TO_VALIDATE) - 1 else None)
fig.tight_layout();

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 3.5))

colors = ["tab:blue", "tab:orange", "tab:green"]
for m, model in enumerate(MODELS):
    average_site_errors = {}
    for i, site in enumerate(UCI_CLIENT_SITES_TO_VALIDATE):
        errors = model_errors[model][site]
        average_site_errors[site] = np.mean(errors)
    
    ax.plot(list(average_site_errors.keys()), list(average_site_errors.values()), label=model, lw=0.75, ls="--", marker="o")
ax.legend()
ax.tick_params(axis="x", rotation=45)
fig.tight_layout();